# Event Study cu Market Model

Acesta este al doilea notebook si trebuie citit dupa `event_study_explicat_ro.ipynb`.

Primul notebook explica modelul exact folosit in codul actual.
Acest notebook explica extensia academica standard: **market model**.

Aici adaugam o idee noua:

- randamentul normal al activului nu este doar o medie constanta
- el depinde si de ce a facut piata in aceeasi zi


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

plt.style.use("ggplot")
pd.set_option("display.float_format", lambda x: f"{x:.6f}")


## 1. De ce avem nevoie de un model mai bogat?

Modelul simplu din primul notebook este:

$$
r_t = \alpha + \varepsilon_t
$$

Dar in practica, daca piata intreaga urca sau cade, este normal ca si activul sa fie influentat.

De aceea, modelul standard din literatura este:

$$
R_{i,t} = \alpha_i + \beta_i R_{m,t} + \varepsilon_{i,t}
$$

Unde:

- $R_{i,t}$ este randamentul activului
- $R_{m,t}$ este randamentul pietei
- $\alpha_i$ este componenta proprie activului
- $\beta_i$ masoara sensibilitatea la piata
- $\varepsilon_{i,t}$ este partea neexplicata de piata


## 2. Exemplu didactic mic

Mai intai lucram cu un set de date mic si controlat, ca sa vezi matematica clar.

- primele 8 observatii: fereastra de estimare
- ultimele 4 observatii: fereastra de eveniment


In [ ]:
dates = pd.date_range("2024-01-02", periods=12, freq="B")

market_returns = pd.Series(
    [0.004, -0.002, 0.006, 0.001, -0.003, 0.005, 0.002, -0.004, 0.003, 0.001, 0.008, -0.007],
    index=dates,
    name="Randament piata",
)

asset_returns = pd.Series(
    [0.006, -0.004, 0.009, 0.002, -0.005, 0.007, 0.004, -0.006, 0.005, 0.003, 0.018, -0.014],
    index=dates,
    name="Randament activ",
)

df = pd.concat([asset_returns, market_returns], axis=1)
display(df)


In [ ]:
estimation_df = df.iloc[:8].copy()
event_df = df.iloc[8:].copy()

fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df.index, df["Randament activ"] * 100, marker="o", linewidth=2, label="Randament activ", color="#1f77b4")
ax.plot(df.index, df["Randament piata"] * 100, marker="s", linewidth=2, label="Randament piata", color="#ff7f0e")
ax.axvspan(estimation_df.index.min(), estimation_df.index.max(), color="#2ca02c", alpha=0.18, label="Estimare")
ax.axvspan(event_df.index.min(), event_df.index.max(), color="#d62728", alpha=0.14, label="Eveniment")
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Datele didactice: activ vs. piata")
ax.set_xlabel("Data")
ax.set_ylabel("Randament (%)")
ax.legend(loc="best")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 3. Cum se antreneaza regresia liniara

Modelul este:

$$
R_{i,t} = \alpha_i + \beta_i R_{m,t} + \varepsilon_{i,t}
$$

Trebuie sa estimam $\alpha_i$ si $\beta_i$ din fereastra de estimare.

Formulele OLS sunt:

$$
\hat{\beta}_i = \frac{\operatorname{Cov}(R_i, R_m)}{\operatorname{Var}(R_m)}
$$

$$
\hat{\alpha}_i = \overline{R_i} - \hat{\beta}_i \overline{R_m}
$$

Interpretarea lui $\beta$:

- daca $\beta = 1$, activul tinde sa se miste cam ca piata
- daca $\beta > 1$, activul reactioneaza mai puternic
- daca $\beta < 1$, activul reactioneaza mai slab


In [ ]:
R_i = estimation_df["Randament activ"]
R_m = estimation_df["Randament piata"]

beta_hat = np.cov(R_i, R_m, ddof=1)[0, 1] / np.var(R_m, ddof=1)
alpha_hat = R_i.mean() - beta_hat * R_m.mean()

display(
    pd.Series(
        {
            "alpha estimat": alpha_hat,
            "alpha estimat (%)": alpha_hat * 100,
            "beta estimat": beta_hat,
        }
    )
)


In [ ]:
x_line = np.linspace(R_m.min(), R_m.max(), 100)
y_line = alpha_hat + beta_hat * x_line

fig, ax = plt.subplots(figsize=(6.5, 5.5))
ax.scatter(R_m * 100, R_i * 100, s=70, color="#1f77b4", label="Observatii istorice")
ax.plot(x_line * 100, y_line * 100, color="#d62728", linewidth=2, label="Linia de regresie")
ax.set_title("Regresia liniara: activ vs. piata")
ax.set_xlabel("Randament piata (%)")
ax.set_ylabel("Randament activ (%)")
ax.legend()
plt.tight_layout()
plt.show()


### Exemplu foarte mic, facut de mana

Daca piata are `[1%, 2%, 3%]` si activul are `[2%, 3%, 4%]`, intuitia este:

- activul urmeaza piata aproape unu-la-unu
- dar este si putin peste ea

O regresie liniara ar produce aproximativ:

- $\beta \approx 1$
- $\alpha \approx 1\%$


In [ ]:
market_toy = np.array([0.01, 0.02, 0.03])
asset_toy = np.array([0.02, 0.03, 0.04])

beta_toy = np.cov(asset_toy, market_toy, ddof=1)[0, 1] / np.var(market_toy, ddof=1)
alpha_toy = asset_toy.mean() - beta_toy * market_toy.mean()

print(f"beta toy = {beta_toy:.3f}")
print(f"alpha toy = {alpha_toy:.3f}")


## 4. Cum se face predictia

Pentru fiecare zi din fereastra de eveniment:

$$
\widehat{R}_{i,t} = \hat{\alpha}_i + \hat{\beta}_i R_{m,t}
$$

Diferenta fata de primul notebook este importanta:

- acolo predictia era constanta
- aici predictia se modifica de la o zi la alta, pentru ca depinde de piata


In [ ]:
event_df = event_df.copy()
event_df["Randament normal prezis"] = alpha_hat + beta_hat * event_df["Randament piata"]
event_df["Randament anormal"] = event_df["Randament activ"] - event_df["Randament normal prezis"]

display(event_df)


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(event_df.index, event_df["Randament activ"] * 100, marker="o", linewidth=2, label="Randament real", color="#1f77b4")
ax.plot(event_df.index, event_df["Randament normal prezis"] * 100, marker="s", linestyle="--", linewidth=2, label="Randament normal prezis", color="#ff7f0e")
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Fereastra de eveniment: real vs. prezis")
ax.set_xlabel("Data")
ax.set_ylabel("Randament (%)")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
car = event_df["Randament anormal"].cumsum()

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

axes[0].bar(event_df.index, event_df["Randament anormal"] * 100, color="#d62728")
axes[0].axhline(0, color="black", linewidth=1)
axes[0].set_title("Randamente anormale (%)")
axes[0].set_ylabel("%")

axes[1].plot(event_df.index, car * 100, marker="o", linewidth=2, color="#9467bd")
axes[1].axhline(0, color="black", linewidth=1)
axes[1].set_title("CAR: cumulul randamentelor anormale (%)")
axes[1].set_ylabel("%")
axes[1].set_xlabel("Data")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 5. `t-test` si `p-value`

Dupa ce avem randamentele anormale, repetam ideea statistica de baza.

Mai intai calculam reziduurile istorice din fereastra de estimare:

$$
\hat{\varepsilon}_{i,t} = R_{i,t} - (\hat{\alpha}_i + \hat{\beta}_i R_{m,t})
$$

Apoi folosim:

$$
t = \frac{\overline{AR}}{s_{\varepsilon} / \sqrt{n}}
$$

unde $s_{\varepsilon}$ este abaterea standard a reziduurilor istorice.


In [ ]:
estimation_residuals = estimation_df["Randament activ"] - (alpha_hat + beta_hat * estimation_df["Randament piata"])
residual_std = estimation_residuals.std(ddof=1)

abnormal_returns = event_df["Randament anormal"]
mean_ar = abnormal_returns.mean()
n = len(abnormal_returns)
t_stat = mean_ar / (residual_std / np.sqrt(n))
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), df=n - 1))

display(
    pd.Series(
        {
            "Media randamentelor anormale": mean_ar,
            "Media randamentelor anormale (%)": mean_ar * 100,
            "Dev. std. reziduuri istorice": residual_std,
            "t-statistic": t_stat,
            "p-value": p_value,
            "Semnificativ la prag de 5%": p_value < 0.05,
        }
    )
)


## 6. Exemplu si cu date reale Yahoo Finance

Acum trecem de la exemplul didactic la un exemplu real.

Alegem:

- activ: `AAPL`
- piata: `^GSPC` (S&P 500)

Aceste tickere sunt alese doar pentru usurinta demonstratiei.

Cell-ul urmator incearca sa descarce date reale. Daca nu merge reteaua sau Yahoo Finance nu raspunde, notebook-ul nu cade: doar afiseaza un mesaj si sari peste sectiunea live.


In [ ]:
import yfinance as yf

asset_ticker = "AAPL"
market_ticker = "^GSPC"
event_date = pd.Timestamp("2020-03-16")
download_start = "2019-01-01"
download_end = "2020-06-30"

real_data_available = False
real_download_message = ""

try:
    asset_close = yf.download(asset_ticker, start=download_start, end=download_end, auto_adjust=True, progress=False)["Close"].rename(asset_ticker)
    market_close = yf.download(market_ticker, start=download_start, end=download_end, auto_adjust=True, progress=False)["Close"].rename(market_ticker)
    real_prices = pd.concat([asset_close, market_close], axis=1).dropna()
    if real_prices.empty:
        raise ValueError("Yahoo a returnat un tabel gol.")
    real_data_available = True
    real_download_message = "Datele reale au fost descarcate cu succes."
except Exception as exc:
    real_prices = None
    real_download_message = f"Descarcarea live nu a mers: {exc}"

print(real_download_message)


In [ ]:
if real_data_available:
    display(real_prices.head())

    normalized = real_prices / real_prices.iloc[0] * 100
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(normalized.index, normalized[asset_ticker], linewidth=2, label=asset_ticker, color="#1f77b4")
    ax.plot(normalized.index, normalized[market_ticker], linewidth=2, label=market_ticker, color="#ff7f0e")
    ax.axvline(event_date, color="#d62728", linestyle="--", linewidth=2, label="Data eveniment")
    ax.set_title("Preturi normalizate la 100: activ vs. piata")
    ax.set_xlabel("Data")
    ax.set_ylabel("Indice normalizat")
    ax.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Sectiunea live este sarita pentru ca nu exista date reale disponibile in aceasta sesiune.")


In [ ]:
if real_data_available:
    real_returns = real_prices.pct_change().dropna()

    estimation_end = event_date - pd.offsets.BDay(3)
    estimation_start = estimation_end - pd.offsets.BDay(119)
    event_start = event_date - pd.offsets.BDay(2)
    event_end = event_date + pd.offsets.BDay(5)

    real_estimation = real_returns.loc[estimation_start:estimation_end].copy()
    real_event = real_returns.loc[event_start:event_end].copy()

    R_i_real = real_estimation[asset_ticker]
    R_m_real = real_estimation[market_ticker]

    beta_real = np.cov(R_i_real, R_m_real, ddof=1)[0, 1] / np.var(R_m_real, ddof=1)
    alpha_real = R_i_real.mean() - beta_real * R_m_real.mean()

    real_event["randament_prezis"] = alpha_real + beta_real * real_event[market_ticker]
    real_event["randament_anormal"] = real_event[asset_ticker] - real_event["randament_prezis"]

    display(
        pd.Series(
            {
                "alpha real": alpha_real,
                "beta real": beta_real,
                "observatii estimare": len(real_estimation),
                "observatii eveniment": len(real_event),
            }
        )
    )
    display(real_event.head())
else:
    print("Nu exista rezultate live de afisat.")


In [ ]:
if real_data_available:
    fig, axes = plt.subplots(3, 1, figsize=(11, 10), sharex=True)

    axes[0].scatter(R_m_real * 100, R_i_real * 100, s=22, color="#1f77b4", alpha=0.8)
    x_line = np.linspace(R_m_real.min(), R_m_real.max(), 100)
    y_line = alpha_real + beta_real * x_line
    axes[0].plot(x_line * 100, y_line * 100, color="#d62728", linewidth=2)
    axes[0].set_title("Estimare live: regresia activ-piata")
    axes[0].set_ylabel("Randament activ (%)")

    axes[1].plot(real_event.index, real_event[asset_ticker] * 100, marker="o", linewidth=2, label="Randament real", color="#1f77b4")
    axes[1].plot(real_event.index, real_event["randament_prezis"] * 100, marker="s", linestyle="--", linewidth=2, label="Randament prezis", color="#ff7f0e")
    axes[1].axhline(0, color="black", linewidth=1)
    axes[1].set_title("Eveniment live: real vs. prezis")
    axes[1].set_ylabel("%")
    axes[1].legend()

    axes[2].bar(real_event.index, real_event["randament_anormal"] * 100, color="#d62728")
    axes[2].axhline(0, color="black", linewidth=1)
    axes[2].plot(real_event.index, real_event["randament_anormal"].cumsum() * 100, marker="o", linewidth=2, color="#9467bd", label="CAR")
    axes[2].set_title("Randamente anormale si CAR")
    axes[2].set_ylabel("%")
    axes[2].set_xlabel("Data")
    axes[2].legend()

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("Graficele live nu sunt disponibile fara date Yahoo Finance.")


## 7. Ce trebuie retinut

Primul notebook:

- explica modelul exact din codul actual
- foloseste doar media randamentelor istorice

Acest al doilea notebook:

- introduce randamentul pietei
- estimeaza `alpha` si `beta`
- produce un randament normal mai realist
- este mai apropiat de forma academica standard din finante

Ordinea buna de invatare ramane:

1. modelul simplu
2. market model
3. apoi legatura cu date reale Yahoo Finance
